# ClusterBP v2: новые признаки — терапия и длинная история

Два недорогих расширения признаков XGB_full:

1. **Терапия** (`Медикаментозная терапия от 10.04.2025.csv`, в v2 не использовалась). Тип B — «давление выше целевого»,
   то есть неэффективная терапия, поэтому смена и усиление лечения должны быть с ним связаны. Признаки на момент отсечки:
   число активных гипотензивных препаратов и классов, классы (ИАПФ, БРА, БКК, ББ, диуретики, центральные, α-блокаторы),
   новые гипотензивные назначения за 14/30/60 дней, отмены за 30 дней, дней с последнего изменения, статин, антиагрегант.
   Учитываются только назначения, день которых **строго раньше** дня отсечки.
2. **Длинная история** (30/60/90 дней). По Rothwell прогностична долговременная (межвизитная) вариабельность САД,
   а история событий — самый сильный признак. Признаки — по среднесуточным значениям: среднее, SD, ARV и наклон САД,
   доля дней со средним САД ≥ 140, отклонение последних 14 дней от долговременного уровня, максимум; число событий A, B
   и всех КЗС за 30/60/90 дней и за всё время в программе.

**Две оговорки по терапии, которые проверяются ниже до моделирования:**
- **покрытие:** около двух записей на пациента; если почти все назначения сделаны при включении в программу, признаков
  «изменения» почти не будет и останутся только статические (число и классы препаратов);
- **обратная причинность:** врач усиливает терапию *после* тревог B, поэтому «недавняя смена терапии» отчасти повторяет
  историю событий. Значим только прирост **поверх** XGB_full, где история событий уже есть.
  Кроме того, неизвестно, когда назначение было внесено в систему: если записи вносились задним числом, признак может
  использовать информацию, которой на момент отсечки ещё не было. Это стоит уточнить у оператора.

Сравнение — те же фолды и фенотипы, что в `clusterbp_v2.ipynb`, XGBoost с ранней остановкой на внутренней валидации,
парный бутстреп по пациентам; отдельно — на окнах без событий этого типа в прошлом (первые события).

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clusterbp_lib import RESULTS_DIR, DATA_DIR, SEED, TARGETS, FEATURE_SETS, N_FOLDS, fold_masks
from clusterbp_extra import (load_windows, phenotypes, fit_predict, compare_table, long_history_features,
                             load_therapy, therapy_features, AH_CLASSES, CLASS_COL)

warnings.filterwarnings('ignore')
OUT = os.path.join(RESULTS_DIR, 'features')
os.makedirs(OUT, exist_ok=True)
F_TARGETS = ['A', 'B']

win, pheno_win, pheno_first, pheno_first_pid = load_windows()
meas = pd.read_parquet(os.path.join(DATA_DIR, 'measurements.parquet'))
events = pd.read_parquet(os.path.join(DATA_DIR, 'events.parquet'))
pid = win['pid'].values
cohort_pids = np.unique(pid)
first_day = meas.groupby('pid')['td'].min().apply(np.floor)

## 1. Терапия: покрытие и время назначений

In [ ]:
th = load_therapy()
th_c = th[th['pid'].isin(cohort_pids)].copy()
th_c['days_from_first_meas'] = th_c['t_known'] - first_day.reindex(th_c['pid'].values).values
in_program = th_c['days_from_first_meas'] > 14            # известно позже, чем через 14 дней после первого измерения
pat_with = th_c.groupby('pid').size()
pat_change = th_c[in_program & th_c['is_AH']].groupby('pid').size()
cov = pd.Series({
    'записей в файле': len(th),
    'записей, связанных с пациентом через id программы': int(th['pid'].notna().sum()),
    'записей у пациентов когорты': len(th_c),
    'пациентов когорты': len(cohort_pids),
    'пациентов когорты с ≥ 1 назначением': len(pat_with),
    'доля пациентов с назначениями': len(pat_with) / len(cohort_pids),
    'назначений на пациента (медиана среди имеющих)': pat_with.median(),
    'доля гипотензивных назначений': th_c['is_AH'].mean(),
    'доля назначений без распознанного класса': (th_c['n_classes'] == 0).mean(),
    'доля назначений, известных > 14 дней после первого измерения': in_program.mean(),
    'пациентов с гипотензивным назначением во время наблюдения': len(pat_change),
    'доля таких пациентов': len(pat_change) / len(cohort_pids),
    'доля записей «принимаемая терапия = да»': th_c['taking'].mean(),
    'доля записей с датой окончания': th_c['t_end'].notna().mean(),
}, name='значение')
cov.to_csv(os.path.join(OUT, 'therapy_coverage.csv'))
display(cov.round(3))

timing = th_c['days_from_first_meas'].describe(percentiles=[.05, .25, .5, .75, .95]).rename('дней от первого измерения до назначения')
display(timing.round(1))
cls = pd.Series({c: th_c[f'is_{c}'].mean() for c in AH_CLASSES + ['статин', 'антиагрегант/антикоагулянт']},
                name='доля назначений').sort_values(ascending=False)
display(cls.round(3))
unknown = th_c.loc[th_c['n_classes'] == 0, 'МНН'].value_counts().head(20)
print('Нераспознанные МНН (проверьте словарь DRUG_CLASSES в clusterbp_extra.py):')
display(unknown)

In [ ]:
TH = therapy_features(win, th)
LH = long_history_features(win, meas, events)
for c in list(TH.columns) + list(LH.columns):
    win[c] = (TH[c] if c in TH else LH[c]).values

# Обратная причинность: связаны ли недавние изменения терапии с прошлыми событиями B
rc = []
for label, m in [('было событие B за 14 дней до отсечки', win['ev_B_14'].values > 0),
                 ('не было событий B за 14 дней', win['ev_B_14'].values == 0)]:
    rc.append({'окна': label, 'окон': int(m.sum()),
               'доля окон с новым гипотензивным назначением за 14 дней': (win.loc[m, 'th_new_ah_14'] > 0).mean(),
               'доля окон с новым назначением за 30 дней': (win.loc[m, 'th_new_ah_30'] > 0).mean()})
rc = pd.DataFrame(rc)
display(rc.round(4))

# Доля событий B в зависимости от терапии — отдельно при наличии и отсутствии недавних событий B
desc = []
for hist_label, hm in [('без событий B за 14 дней', win['ev_B_14'].values == 0),
                       ('с событиями B за 14 дней', win['ev_B_14'].values > 0)]:
    for label, m in [('нет записей о терапии', win['th_has_record'].values == 0),
                     ('0 гипотензивных классов', (win['th_has_record'].values == 1) & (win['th_n_ah_classes'].values == 0)),
                     ('1 класс', win['th_n_ah_classes'].values == 1), ('2 класса', win['th_n_ah_classes'].values == 2),
                     ('≥ 3 классов', win['th_n_ah_classes'].values >= 3),
                     ('новое гипотензивное назначение за 30 дней', win['th_new_ah_30'].values > 0)]:
        mm = hm & m
        desc.append({'история': hist_label, 'терапия': label, 'окон': int(mm.sum()),
                     'доля событий B': win['y_B'].values[mm].mean() if mm.any() else np.nan,
                     'доля событий A': win['y_A'].values[mm].mean() if mm.any() else np.nan})
desc = pd.DataFrame(desc)
pd.concat([rc.assign(таблица='обратная причинность'), desc.assign(таблица='доля событий по терапии')]) \
    .to_csv(os.path.join(OUT, 'therapy_descriptive.csv'), index=False)
display(desc.round(3))

## 2. Прирост к XGB_full

Варианты: XGB_full; + терапия; + длинная история (только давление); + длинная история (давление и события);
+ терапия и вся длинная история. Оценка на всех окнах и на окнах без событий этого типа в прошлом.

In [ ]:
TH_COLS = list(TH.columns)
LH_BP = [c for c in LH.columns if '_ev_' not in c]
LH_EV = [c for c in LH.columns if '_ev_' in c]
FULL = FEATURE_SETS['full']
VARIANTS = {
    'XGB_full': FULL,
    '+ терапия': FULL + TH_COLS,
    '+ длинная история АД': FULL + LH_BP,
    '+ длинная история АД и событий': FULL + LH_BP + LH_EV,
    '+ терапия и длинная история': FULL + TH_COLS + LH_BP + LH_EV,
}
scores = {t: {v: (np.full(len(win), np.nan), np.zeros(len(win), np.int8)) for v in VARIANTS} for t in F_TARGETS}
importance = []
t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    phenotypes(win, pheno_win, pheno_first, pheno_first_pid, k)
    for t in F_TARGETS:
        y = win[f'y_{t}'].values
        for v, cols in VARIANTS.items():
            s, thr, m = fit_predict(win, cols, y, tr, va, SEED + k)
            sc, pr = scores[t][v]
            sc[te] = s[te]
            pr[te] = s[te] > thr
            if v == '+ терапия и длинная история':
                importance.append(pd.Series(m.feature_importances_, index=cols, name=f'{t}_fold{k}'))
    print(f'фолд {k}: {(time.time() - t_all) / 60:.1f} мин')

In [ ]:
tables = []
for t in F_TARGETS:
    y = win[f'y_{t}'].values
    for subset, m in [('все окна', np.ones(len(win), bool)),
                      (f'без событий {t} до отсечки', np.isnan(win[f'ev_{t}_days_since'].values))]:
        tab = compare_table({v: (m, y, *scores[t][v]) for v in VARIANTS}, pid, ref='XGB_full')
        tab.insert(0, 'окна', subset)
        tab.insert(0, 'target', t)
        tables.append(tab)
gain = pd.concat(tables)
gain.to_csv(os.path.join(OUT, 'feature_gain.csv'))
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
display(gain[['target', 'окна', 'доля событий', 'AUROC', 'AUPRC', 'AUPRC 95% ДИ', 'ΔAUPRC', 'ΔAUPRC 95% ДИ', 'p ΔAUPRC',
              'macro_F1', 'Δmacro_F1', 'Δmacro_F1 95% ДИ']].round(4))

## 3. Важность новых признаков (gain XGBoost, среднее по фолдам)

In [ ]:
imp = pd.concat(importance, axis=1)
rows = []
for t in F_TARGETS:
    s = imp[[c for c in imp.columns if c.startswith(f'{t}_')]].mean(1)
    s = s / s.sum()
    grp = pd.Series({'терапия': s[TH_COLS].sum(), 'длинная история АД': s[LH_BP].sum(),
                     'длинная история событий': s[LH_EV].sum(),
                     'история событий 7–14 дней': s[[c for c in FULL if c.startswith('ev_')]].sum(),
                     'давление 14 дней, статика, фенотип': s[[c for c in FULL if not c.startswith('ev_') and c in s]].sum()})
    rows.append(grp.rename(t))
    top = s.sort_values(ascending=False).head(20)
    top.to_csv(os.path.join(OUT, f'importance_top20_{t}.csv'))
    print(f'\n{t}: 20 самых важных признаков')
    display(top.round(4))
groups = pd.DataFrame(rows)
groups.to_csv(os.path.join(OUT, 'importance_groups.csv'))
display(groups.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
groups.T.plot.bar(ax=ax)
ax.set_ylabel('доля суммарного gain')
ax.set_title('XGB_full + терапия + длинная история: вклад групп признаков')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'importance_groups.png'), dpi=200, bbox_inches='tight')
plt.show()